In [1]:
import pandas as pd
import json
import ast
import os

In [2]:
soc_crosswalk = pd.read_csv('../data/labor_codes/2019_to_SOC_Crosswalk.csv')
code_title_map = dict(zip(
    soc_crosswalk['O*NET-SOC 2019 Code'], 
    soc_crosswalk['O*NET-SOC 2019 Title']
))

In [3]:
WORLDBANK_INPUT_URL = '../data/ai_measurements/worldbank_metrics/input/'
WORLDBANK_OUTPUT_URL = '../data/ai_measurements/worldbank_metrics/output/'

# 1. Calculate AIOE

The point of this folder is to calculate the AIOE from the study of [Felten et al. (2021)](https://sms.onlinelibrary.wiley.com/doi/full/10.1002/smj.3286?__cf_chl_tk=PsCX9B8s7IxRu787dWHte8A8ER.iEVyrfuvdIs9d5e0-1788675243-1.0.1.1-HORPGuasZa2eX7ohUBzY56irDllfBHlxA2oChVTgVHQ). To have an interesting analysis, we will compute the AIOE for three time periods of 

In [4]:
# get the exposure of each ability to AI
ability_exposure_df = pd.read_excel(
    os.path.join(WORLDBANK_INPUT_URL, 'AIOE_DataAppendix.xlsx'), 
    sheet_name='Appendix D',
    index_col=0)
ability_exposure_df = ability_exposure_df.sum(axis=1)
ability_exposure_map = ability_exposure_df.to_dict()

# standardize Visual Color Determination to Visual Color Discrimination
ability_exposure_map['Visual Color Discrimination'] = ability_exposure_map['Visual Color Determination']

# get the abilities data
abilities_df = pd.read_csv(
    os.path.join(WORLDBANK_INPUT_URL, 'abilities.csv')
)
abilities_df = abilities_df.pivot(
    index=['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name'], 
    columns='Scale Name', 
    values='Data Value'
).reset_index(drop=False)

To account for SOC codes without available data, mean imputation will be performed using neighboring SOC codes, defined as occupations sharing the same first five SOC digits. In total, 67 of the 910 SOC codes (7.4%) are missing data and will therefore be imputed using this approach.

In [5]:
# read data
mca_df = pd.read_csv('../data/FINAL_mca_data.csv')
mca_df['SOC Codes'] = mca_df['SOC Codes'].apply(ast.literal_eval)

# gather the codes that have missing data
all_possible_codes = set(code_title_map.keys())
existing_codes = set(abilities_df['O*NET-SOC Code'])
missing_codes = all_possible_codes - existing_codes

# create the prefixes
first_n_prefix = 4
abilities_df['SOC Prefix'] = (
    abilities_df['O*NET-SOC Code']
    .str.replace('-', '', regex=False)
    .str[:first_n_prefix]
)

missing_codes_df = pd.DataFrame({
    'O*NET-SOC Code': list(missing_codes)
})

missing_codes_df['SOC Prefix'] = (
    missing_codes_df['O*NET-SOC Code']
    .str.replace('-', '', regex=False)
    .str[:first_n_prefix]
)

# Calculate the mean ability profile for each 5-digit group
ability_means = (
    abilities_df
    .groupby(['SOC Prefix', 'Element ID', 'Element Name'])[
        ['Importance', 'Level']
    ]
    .mean()
    .reset_index()
)

# Create the imputed rows
abilities_to_impute = (
    missing_codes_df
    .merge(
        ability_means,
        on='SOC Prefix',
        how='left'
    )
)
abilities_to_impute['Title'] = abilities_to_impute['O*NET-SOC Code'].map(code_title_map)

# merge the partial and missing df
complete_abilities_df = pd.concat([abilities_df, abilities_to_impute])

In [6]:
# set the exposure
for ability, exposure in ability_exposure_map.items():
    is_ability = (complete_abilities_df['Element Name'] == ability)
    complete_abilities_df.loc[is_ability, 'Exposure'] = exposure

# calculate the exposure per job per exposure
complete_abilities_df['Importance-Weighted Level'] = complete_abilities_df.Importance * complete_abilities_df.Level
complete_abilities_df['Exposure Weighted'] = complete_abilities_df.Exposure * complete_abilities_df['Importance-Weighted Level']

abilities_exposure_df = complete_abilities_df.groupby('O*NET-SOC Code')[['Exposure Weighted', 'Importance-Weighted Level']].sum()
abilities_exposure_df['AIOE'] = abilities_exposure_df['Exposure Weighted'] / abilities_exposure_df['Importance-Weighted Level']

# mean center AIOE
AIOE_mean = abilities_exposure_df.AIOE.mean()
AIOE_std = abilities_exposure_df.AIOE.std()
abilities_exposure_df['Standardized AIOE'] = (abilities_exposure_df.AIOE - AIOE_mean) / AIOE_std

# output it as a json
abilities_exposure_df['AIOE'].to_json(os.path.join(WORLDBANK_OUTPUT_URL, 'soc_raw_aioe.json'))
abilities_exposure_df['Standardized AIOE'].to_json(os.path.join(WORLDBANK_OUTPUT_URL, 'soc_standardized_aioe.json'))

# 2. Calculate Complementarity

Pizzinelli et al. (2023) extend the AIOE framework by introducing a complementarity index that measures the extent to which AI supports rather than substitutes human labor. This index is higher for occupations where AI is more likely to enhance human abilities, taking into account both the relevance of AI applications and the broader social and technical contexts that may limit automation.

The authors construct the complementarity index using 11 of the 57 O*NET work context variables together with job zones, grouped into six dimensions. For each occupation in the SOC system, the work context variables provide a score from 0 to 100, which reflects how important or frequent a given task or condition is for that job (e.g., the importance of face-to-face discussions, the frequency of decision-making, or the degree of automation). Job Zone, by contrast, is an ordinal variable ranging from 1 to 5, which indicates the preparation time required to gain the skills for an occupation. In the index, Job Zone values are rescaled by multiplying by 20, producing a 0–100 scale consistent with the other variables. The six grouped dimensions with their work contexts are:

1. *Communication* – Face-to-Face Discussions, Public Speaking

2. *Responsibility* – Responsibility for Outcomes, Responsibility for Others’ Health

3. *Physical Conditions* – Exposure to Outdoor Environments, Physical Proximity to Others

4. *Criticality* – Consequence of Errors, Freedom of Decisions, Frequency of Decisions

5. *Routine* – Degree of Automation\*, Structured vs. Unstructured Work

6. *Skills* – proxied by Job Zone 

For each dimension, an occupation’s score is calculated as the average of its component work context variables. For example, the *Communication* dimension will have its score as the average of the score from Face-to-Face Discussions and Public Speaking. Finally, the complementarity index is obtained by taking the average across the six dimensions, providing a single measure of how strongly AI complements a given occupation.

---
\*As a note, the degree of automation was inverted because it is the only work context in which higher values indicate lower complementarity with AI. To address this, we created a new variable called degree of freedom, which is derived from the degree of automation. In this formulation, higher scores correspond to greater complementarity with AI. Formally, $$\text{Degree of Freedom} = 100 - \text{Degree of Automation}$$.

In [7]:
work_contexts_df = pd.read_csv(os.path.join(WORLDBANK_INPUT_URL, 'work_context.csv'))
work_contexts_df = work_contexts_df[work_contexts_df['Scale ID'] == 'CX'].copy()
work_contexts_df['Data Value'] = (work_contexts_df['Data Value'] - 1) * 25

# create the degre of freedom
work_contexts_df['Element Name'] = work_contexts_df['Element Name'].replace({'Degree of Automation':'Degree of Freedom'})
is_degree_freedom = work_contexts_df['Element Name'] == 'Degree of Freedom'
work_contexts_df.loc[is_degree_freedom, 'Data Value'] = 100 - work_contexts_df.loc[is_degree_freedom, 'Data Value']

# use only the relevant work contexts
relevant_contexts = [
    "Face-to-Face Discussions with Individuals and Within Teams",
    "Public Speaking",
    "Work Outcomes and Results of Other Workers",
    "Health and Safety of Other Workers",
    "Determine Tasks, Priorities and Goals",
    "Outdoors, Exposed to All Weather Conditions",
    "Physical Proximity",
    "Consequence of Error",
    "Frequency of Decision Making",
    "Degree of Freedom",
    "Freedom to Make Decisions",
]

is_relevant_context = work_contexts_df['Element Name'].isin(relevant_contexts)
work_contexts_df = work_contexts_df[is_relevant_context]
work_contexts_df = work_contexts_df[['O*NET-SOC Code', 'Title', 'Element Name', 'Data Value']].copy()

In [8]:
# get also the job zones
job_zones_df = pd.read_csv(os.path.join(WORLDBANK_INPUT_URL, 'job_zones.csv'))
job_zones_df['Data Value'] = job_zones_df['Job Zone'] * 20
job_zones_df['Element Name'] = 'Skills' 
job_zones_df = job_zones_df[['O*NET-SOC Code', 'Title', 'Element Name', 'Data Value']].copy()

# concatenate them
complementarity_df = pd.concat([work_contexts_df, job_zones_df])
complementarity_df = complementarity_df.pivot(
    index=['O*NET-SOC Code', 'Title'], 
    columns='Element Name', 
    values='Data Value'
).reset_index()

In [9]:
# Create a copy of the complementarity data
complete_complementarity_df = complementarity_df.dropna().copy()

# Set the number of SOC digits to use for matching
first_n_prefix = 4

# Create SOC prefixes
complete_complementarity_df['SOC Prefix'] = (
    complete_complementarity_df['O*NET-SOC Code']
    .str.replace('-', '', regex=False)
    .str[:first_n_prefix]
)

# Identify completely missing SOC codes
all_possible_codes = set(code_title_map.keys())
existing_codes = set(complete_complementarity_df['O*NET-SOC Code'])
missing_codes = all_possible_codes - existing_codes
missing_codes_df = pd.DataFrame({
    'O*NET-SOC Code': list(missing_codes)
})

# Create SOC prefixes for the missing codes
missing_codes_df['SOC Prefix'] = (
    missing_codes_df['O*NET-SOC Code']
    .str.replace('-', '', regex=False)
    .str[:first_n_prefix]
)

# Define the complementarity dimensions
complementarity_columns = relevant_contexts + ['Skills']

# Calculate the mean complementarity profile for each SOC prefix
complementarity_means = (
    complete_complementarity_df
    .groupby('SOC Prefix')[complementarity_columns]
    .mean()
    .reset_index()
)

# Create rows for completely missing SOC codes
missing_complementarity = (
    missing_codes_df
    .merge(
        complementarity_means,
        on='SOC Prefix',
        how='left'
    )
)

# Combine the completely missing SOC codes
complete_complementarity_df = pd.concat(
    [
        complete_complementarity_df,
        missing_complementarity
    ],
    ignore_index=True
)

# Remove the temporary SOC Prefix column
complete_complementarity_df.drop(
    columns='SOC Prefix',
    inplace=True
)
complete_complementarity_df.Title = complete_complementarity_df['O*NET-SOC Code'].map(code_title_map)

# actually compute complementarity
complete_complementarity_df['Complementarity'] = complete_complementarity_df[complementarity_columns].mean(axis=1)
complete_complementarity_df.index = complete_complementarity_df['O*NET-SOC Code']
complete_complementarity_df['Complementarity'].to_json(os.path.join(WORLDBANK_OUTPUT_URL, 'soc_complementarity.json'))

# 3. Calculate C-AIOE

We define C-AIOE for some job $j$ as:
$$\text{C-AIOE}_j = \text{AIOE}_j \times \big(1 - (\theta_j - \theta_{\text{MIN}}) \big)$$

In [ ]:
worldbank_df = (
    pd.concat([abilities_exposure_df, complete_complementarity_df], axis=1)
    .drop(columns=['O*NET-SOC Code'])
)

comple = worldbank_df['Complementarity'] / 100
min_comple = comple.min()
worldbank_df['C-AIOE'] = worldbank_df['AIOE'] * (1 - (comple - min_comple))

worldbank_df = worldbank_df[['Title', 'AIOE', 'Standardized AIOE', 'Complementarity', 'C-AIOE']]

worldbank_df.to_csv(os.path.join(WORLDBANK_OUTPUT_URL, 'soc_aioe_comple.csv'))